In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
import glob
from tqdm import tqdm
import numpy as np
from natsort import natsorted

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10 * 2.54})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}"
full_column_width = 18.2
half_column_width = 8.89

In [ ]:
import jax
import jax.numpy as jnp

gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

In [ ]:
from dmpe.data_management import DataPaths

from dmpe.utils.density_estimation import build_grid
from dmpe.models.models import NeuralEulerODE, NeuralEulerODEPendulum, NeuralEulerODECartpole
from dmpe.models.model_utils import simulate_ahead_with_env
from dmpe.evaluation.plotting_utils import plot_sequence
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results
from dmpe.evaluation.exp_data_model_learning import train_model_on_experiment_data, ModelExpDataResult
from dmpe.evaluation.model_evaluation import ModelWrapper, ModelEvaluator, PredictionComparison, NodeModelWrapper, EnvWrapper
from dmpe.evaluation.utils import default_constraint_function

from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

from dmpe.utils.sets.shared import load_results, DiscretizedSet, SlicedSet, load_discretized_set, save_discretized_set

In [ ]:
from enum import Enum
class Systems(Enum):
    FLUID_TANK = 1
    PENDULUM = 2
    CART_POLE = 3

In [ ]:
S_xu_dict = {
    Systems.FLUID_TANK: load_discretized_set(DataPaths().reach_ci_experiments / "fluid_tank_S_xu_666a769d-8c1b-4b.json"),
    Systems.PENDULUM: load_discretized_set(DataPaths().reach_ci_experiments / "pendulum_S_xu_02430b86-ae0d-42.json"),
    Systems.CART_POLE: load_discretized_set(DataPaths().reach_ci_experiments / "cart_pole_S_xu_8744b5d5-30e6-4b.json"),
}

## multidim coverage

### Density:

In [ ]:
def plot_feature_combinations(data, labels, mode="plot", points_per_dim=100, bandwidth=0.05, all_levels=None, scaling=2/3):
    """Plot all combinations of the data set."""
    assert data.shape[-1] == len(labels)
    assert data.ndim == 2

    n_features = data.shape[-1]

    fig, axs = plt.subplots(nrows=n_features, ncols=n_features, figsize=(half_column_width * scaling, half_column_width* scaling), sharex=True, sharey=True)
    
    for i in range(n_features):
        for j in range(n_features):
            if mode == "plot":
                axs[j, i].scatter(data[..., i], data[..., j], s=0.1)
            elif mode == "contourf":
                density_estimate = DensityEstimate.from_dataset(
                    jnp.concatenate([data[..., i][..., None], data[..., j][..., None]], axis=-1)[None],
                    points_per_dim=points_per_dim,
                    bandwidth=bandwidth,
                )

                p_est = density_estimate.p # / jnp.max(density_estimate.p)
                x = density_estimate.z_g                
                grid_len_per_dim = int(np.sqrt(x.shape[0]))
                x_plot = np.array(x.reshape((grid_len_per_dim, grid_len_per_dim, 2)))
                cax = axs[j, i].contourf(
                    x_plot[..., 0],
                    x_plot[..., 1],
                    p_est.reshape(x_plot.shape[:-1]),
                    antialiased=False,
                    levels=50 if all_levels is None else all_levels[i, j, :],
                    alpha=0.9,
                    cmap=plt.cm.coolwarm,
                )

            axs[j, 0].set_ylabel(labels[j])

            axs[j, i].grid(True)
            axs[j, i].set_xlim(-1.02, 1.02)
            axs[j, i].set_ylim(-1.02, 1.02)                
        
        axs[-1, i].set_xlabel(labels[i])
    #fig.tight_layout(pad=-0.15)

    return fig

In [ ]:
plot_feature_combinations

In [ ]:
sys_name = Systems.CART_POLE

In [ ]:
if sys_name == Systems.FLUID_TANK:
    env, penalty_function, featurize, _ = setup_fluid_tank_env()
    labels = [r"$h$", r"$q_\mathrm{in}$"]
elif sys_name == Systems.PENDULUM:
    env, penalty_function, featurize, _ = setup_pendulum_env()
    labels = [r"$\theta$", r"$\omega$", r"$T$"]
elif sys_name == Systems.CART_POLE:
    env, penalty_function, featurize, _ = setup_cart_pole_env()
    labels = [r"$d$", r"$v$", r"$\theta$", r"$\omega$", r"$F$"]

In [ ]:
result_paths = glob.glob(str(DataPaths().model_learning_cs_out / sys_name.name.lower() / "2step" ) + "/*.eqx")
print("# of results: ", len(result_paths))
result_paths = natsorted(result_paths)

In [ ]:
# this should exist in the PMSM files:

In [ ]:
result = ModelExpDataResult.from_file(
    filename=result_paths[255], # 2
    model_class=NeuralEulerODE,
)

In [ ]:
data_points = jnp.concatenate([result.observations, result.actions], axis=-1)

In [ ]:
fig = plot_feature_combinations(
    data_points,
    labels=labels,
    mode="contourf",
    points_per_dim=20,
    bandwidth=0.08,
    scaling=1,
);

### Diced fill:

In [ ]:
def visualize(
    set_to_visualize,
    reduction_method = jnp.mean,
    labels: None | list[str] = None,
    use_contourf: bool = True,
    grid_spacing: float = 0.0,
    scaling: float = 1.0,
):
    # if len(set_to_visualize.unflattened_shape) == 1:
    #     fig, axs = plt.subplots(1, 1, figsize=(half_column_width, half_column_width))
    #     axs.plot(set_to_visualize.grid, set_to_visualize.mask)
    #     return fig, axs
    # elif len(set_to_visualize.unflattened_shape) == 2:
    #     fig, axs = plt.subplots(1, 1, figsize=(half_column_width, half_column_width))
    #     if use_contourf:
    #         axs.contourf(
    #             set_to_visualize.grid_unflattened[..., 0],
    #             set_to_visualize.grid_unflattened[..., 1],
    #             set_to_visualize.mask_unflattened,
    #         )
    #     else:
    #         # how to change?! -> the extend is half a grid spacing longer in each direction!?
    #         extent = 1 + grid_spacing
    #         axs.imshow(set_to_visualize.mask_unflattened.T, origin="lower", extent=[-extent, extent, -extent, extent])
    #     return fig, axs
    # else:
    dim = set_to_visualize.grid.shape[-1]
    fig, axs = plt.subplots(nrows=dim, ncols=dim, figsize=(half_column_width* scaling, half_column_width* scaling), sharex=True, sharey=True)
    feature_indices = jnp.arange(0, dim, 1).tolist()

    if labels is None:
        labels = jnp.arange(0, dim, 1).tolist()

    for i in range(dim):
        for j in range(dim):

            axs[j, i].grid(True)

            reduction_indices = [f_idx for f_idx in feature_indices if not (f_idx == i or f_idx == j)]
            if len(reduction_indices) == dim - 1:
                continue

            reduced_safe = reduction_method(set_to_visualize.mask_unflattened, axis=tuple(reduction_indices))

            if i > j:
                reduced_safe = jnp.transpose(reduced_safe)

            if use_contourf:
                axs[j, i].contourf(
                    set_to_visualize.grid_unflattened[..., *[0 for _ in range(dim - 2)], 0],
                    set_to_visualize.grid_unflattened[..., *[0 for _ in range(dim - 2)], 1],
                    reduced_safe,
                    vmin=0.0,
                    vmax=1.0,
                )
            else:
                extent = 1 + grid_spacing
                axs[j, i].imshow(reduced_safe.T, origin="lower", extent=[-extent, extent, -extent, extent])
            axs[j, 0].set_ylabel(labels[j])

        axs[-1, i].set_xlabel(labels[i])
        # fig.tight_layout()
    return fig, axs

In [ ]:
sys_name = Systems.CART_POLE

In [ ]:
if sys_name == Systems.FLUID_TANK:
    env, penalty_function, featurize, _ = setup_fluid_tank_env()
    S_xu = S_xu_dict[sys_name]
    labels = [r"$\tilde{h}$", r"$\tilde{q}_\mathrm{in}$"]
    scaling=0.65
elif sys_name == Systems.PENDULUM:
    env, penalty_function, featurize, _ = setup_pendulum_env()
    S_xu = S_xu_dict[sys_name]
    labels = [r"$\tilde{\theta}$", r"$\tilde{\omega}$", r"$\tilde{T}$"]
    scaling=0.75
elif sys_name == Systems.CART_POLE:
    env, penalty_function, featurize, _ = setup_cart_pole_env()
    S_xu = S_xu_dict[sys_name]
    labels = [r"$\tilde{d}$", r"$\tilde{v}$", r"$\tilde{\theta}$", r"$\tilde{\omega}$", r"$\tilde{F}$"]
    scaling=1.0

fig, axs = visualize(S_xu, use_contourf=False)

In [ ]:
result_paths = glob.glob(str(DataPaths().model_learning_cs_out / sys_name.name.lower() / "2step" ) + "/*.eqx")
print("# of results: ", len(result_paths))
result_paths = natsorted(result_paths)

In [ ]:
for sys_name in Systems:
    if sys_name == Systems.FLUID_TANK:
        env, penalty_function, featurize, _ = setup_fluid_tank_env()
        S_xu = S_xu_dict[sys_name]
        labels = [r"$\tilde{h}$", r"$\tilde{q}_\mathrm{in}$"]
        scaling=0.65
        result_idx = 15+25

    elif sys_name == Systems.PENDULUM:
        env, penalty_function, featurize, _ = setup_pendulum_env()
        S_xu = S_xu_dict[sys_name]
        labels = [r"$\tilde{\theta}$", r"$\tilde{\omega}$", r"$\tilde{T}$"]
        scaling=0.75
        result_idx = 15+25
        
    elif sys_name == Systems.CART_POLE:
        env, penalty_function, featurize, _ = setup_cart_pole_env()
        S_xu = S_xu_dict[sys_name]
        labels = [r"$\tilde{d}$", r"$\tilde{v}$", r"$\tilde{\theta}$", r"$\tilde{\omega}$", r"$\tilde{F}$"]
        scaling=1.0
        result_idx = 15+25

    result_paths = glob.glob(str(DataPaths().model_learning_cs_out / sys_name.name.lower() / "2step" ) + "/*.eqx")
    result_paths = natsorted(result_paths)

    result = ModelExpDataResult.from_file(
        filename=result_paths[result_idx],
        model_class=NeuralEulerODE,
    )


    data_points = jnp.concatenate([result.observations, result.actions], axis=-1)
    print(data_points.shape[0])
    
    fig, axs = visualize(S_xu, use_contourf=True, labels=labels, scaling=scaling)
    
    # if sys_name == Systems.FLUID_TANK:
    #     print(result.observations.shape)
    #     print(result.actions.shape)
        
    #     axs.scatter(result.observations, result.actions, c='magenta', edgecolors='white', s=2, linewidths=0.7)
    #     # axs.scatter(S_xu.grid[:, 0], S_xu.grid[:, 1], s=4, c="r")
    # else:
    n_features = env.reset(env.env_properties)[0].shape[-1] + env.action_dim
    
    for i in range(n_features):
        for j in range(n_features):
            axs[j, i].scatter(data_points[..., i], data_points[..., j], c='crimson', s=2),# edgecolors='white', s=4, linewidths=0.1)
    
    plt.savefig(f"data_over_S_xu_{str(sys_name.name).lower()}.png", bbox_inches="tight", dpi=300)
    plt.show()

In [ ]:
# grid = S_xu.grid
if sys_name == Systems.CART_POLE:
    points_per_dim = 7 
elif sys_name == Systems.PENDULUM:
    points_per_dim = 15
else:
    points_per_dim = 25
dim = S_xu.grid.shape[-1]
grid = build_grid(dim, -1, 1, points_per_dim)

grid_spacing = jnp.abs(grid[0] - grid[1])[-1] / 2

In [ ]:
def check_point_to_point(grid_point, point, grid_spacing):
    lower = grid_point - grid_spacing
    upper = grid_point + grid_spacing

    return jnp.logical_and(jnp.all(lower < point), jnp.all(upper > point))

def check_point_to_grid(grid, point, grid_spacing):
    return eqx.filter_vmap(check_point_to_point, in_axes=(0, None, None))(grid, point, grid_spacing)

def check_points_to_grid(grid, points, grid_spacing):
    return eqx.filter_vmap(check_point_to_grid, in_axes=(None, 0, None))(grid, points, grid_spacing)

In [ ]:
if sys_name == Systems.CART_POLE:
    chunk_size = 100
    out = []
    n = data_points.shape[0]
    for i in tqdm(jnp.arange(0, n, chunk_size)):
        out.append(jnp.any(check_points_to_grid(grid, data_points[i : min(i + chunk_size, n)], grid_spacing), axis=0))
    out = jnp.any(jnp.vstack(out), axis=0)
else:
    out = jnp.any(check_points_to_grid(grid, data_points, grid_spacing), axis=0)

In [ ]:
filled_set = DiscretizedSet(grid=grid, mask=out, unflattened_shape=tuple([points_per_dim] * dim))

fig, axs = visualize(filled_set, labels=labels)
plt.show()

fig, axs = visualize(filled_set, labels=labels)
if sys_name == Systems.FLUID_TANK:
    axs.scatter(result.observations, result.actions, s=4)
    # axs.scatter(grid[:, 0], grid[:, 1], s=4, c="r")
else:
    n_features = env.reset(env.env_properties)[0].shape[-1] + env.action_dim
    
    for i in range(n_features):
        for j in range(n_features):
            axs[j, i].scatter(data_points[..., i], data_points[..., j], s=0.25, c="r")
plt.show()

fig, axs = visualize(filled_set, labels=labels, use_contourf=False, grid_spacing=grid_spacing)
plt.show()

fig, axs = visualize(filled_set, labels=labels, use_contourf=False, grid_spacing=grid_spacing)
if sys_name == Systems.FLUID_TANK:
    axs.scatter(result.observations, result.actions, s=4)
    # axs.scatter(grid[:, 0], grid[:, 1], s=4, c="r")
else:
    n_features = env.reset(env.env_properties)[0].shape[-1] + env.action_dim
    
    for i in range(n_features):
        for j in range(n_features):
            axs[j, i].scatter(data_points[..., i], data_points[..., j], s=0.25, c="r")

## model error plots

In [ ]:
result_paths = glob.glob(str(DataPaths().model_learning_cs_out / "fluid_tank" / "2step" ) + "/*.eqx")
print("# of results: ", len(result_paths))
result_paths = natsorted(result_paths)

env, _, featurize, _ = setup_fluid_tank_env()
wrapped_env = EnvWrapper(env, featurize=lambda x: x)

points_per_dim = 100

model_evaluator = ModelEvaluator(
    constraint_function=default_constraint_function,
    gt_model=wrapped_env,
    obs_dim=1,
    act_dim=1,
    validation_points_per_dim=points_per_dim,
    tau=env.tau,
)

In [ ]:
result = ModelExpDataResult.from_file(
    filename=result_paths[222], # 2
    model_class=NeuralEulerODE,
)
print("Number of datapoints in underlying dataset:", result.n_datapoints)

fig, axs = visualize_model_prediction_performance(
    NodeModelWrapper(result.median_model, featurize=wrapped_env.featurize),
    model_evaluator,
    labels=["h", "q_in"],
)

data_points = jnp.concatenate([result.observations, result.actions], axis=-1)
n_features = data_points.shape[-1]

for i in range(n_features):
    for j in range(n_features):
        axs[j, i].scatter(data_points[..., i], data_points[..., j], s=1, c="crimson")

plt.show()

- what is the magnitude of these errors though?

In [ ]:
result_paths = glob.glob(str(DataPaths().model_learning_cs_out / "cart_pole" / "2step") + "/*.eqx")
print("# of results: ", len(result_paths))
result_paths = natsorted(result_paths)

env, penalty_function, featurize, _ = setup_cart_pole_env()

# featurize = lambda x: x
wrapped_env = EnvWrapper(env, featurize=featurize)

points_per_dim = 20

model_evaluator = ModelEvaluator(
    constraint_function=default_constraint_function,
    gt_model=wrapped_env,
    obs_dim=4,
    act_dim=1,
    validation_points_per_dim=points_per_dim,
    tau=env.tau,
)

In [ ]:
def visualize_model_prediction_performance(wrapped_model, model_evaluator: ModelEvaluator, labels: list[str]):
    difference_map, _ = model_evaluator.default_metrics["pred_comp"](
        wrapped_model,
        model_evaluator.gt_model,
    )

    n_features = model_evaluator.obs_dim + model_evaluator.act_dim
    reshaped_difference_map = difference_map.reshape([model_evaluator.validation_points_per_dim] * n_features + [-1])
    # abs_map = jnp.mean(jnp.abs(reshaped_difference_map) ** 2, axis=-1)
    abs_map = jnp.linalg.norm(reshaped_difference_map, axis=-1)

    fig, axs = plt.subplots(nrows=n_features, ncols=n_features, figsize=(half_column_width, half_column_width), sharex=True, sharey=True)

    feature_indices = jnp.arange(0, n_features, 1).tolist()

    for i in range(n_features):
        for j in range(n_features):

            axs[j, i].grid(True)
            axs[j, i].set_xlim(-1.1, 1.1)
            axs[j, i].set_ylim(-1.1, 1.1)

            reduction_indices = [f_idx for f_idx in feature_indices if not (f_idx == i or f_idx == j)]
            if len(reduction_indices) == n_features - 1:
                continue

            image = jnp.mean(jnp.abs(abs_map), axis=tuple(reduction_indices))

            if i < j:
                image = jnp.transpose(image)

            # TODO: replace with contourf?
            axs[j, i].imshow(image, origin="lower", extent=[-1, 1, -1, 1])
            axs[j, 0].set_ylabel(labels[j])

        axs[-1, i].set_xlabel(labels[i])

    return fig, axs

In [ ]:
result = ModelExpDataResult.from_file(
    filename=result_paths[5],
    model_class=NeuralEulerODECartpole,
)

fig, axs = visualize_model_prediction_performance(
    NodeModelWrapper(result.median_model, featurize=featurize),
    model_evaluator,
    labels=[r"$d$", r"$v$", r"$\theta$", r"$\omega$", r"$F$"],
)

data_points = jnp.concatenate([result.observations, result.actions], axis=-1)
n_features = data_points.shape[-1]

for i in range(n_features):
    for j in range(n_features):
        axs[j, i].scatter(data_points[..., i], data_points[..., j], s=0.25, c="r")
axs[j, i].grid(True)
axs[j, i].set_xlim(-1.1, 1.1)
axs[j, i].set_ylim(-1.1, 1.1)

plt.savefig(f"qual_model_error_cart_pole_random_walk.pdf", bbox_inches='tight');
plt.show()